##### Nombre : Daana Valeria Alanis De La Cerda
##### Matrícula : 606878


# Análisis Exploratorio de Datos con el dataset de diamantes

En este notebook aprenderás a realizar un análisis exploratorio de datos (EDA) utilizando una base de datos SQLite con información sobre diamantes. Exploraremos tipos de variables, escalas de medición, estadísticas descriptivas, visualizaciones y transformaciones útiles para preparar los datos para modelos de machine learning.

---

### Conexión a la base de datos

### ¿Qué es Git?

Git es una herramienta que permite gestionar versiones de archivos, especialmente útil en proyectos de programación. Con Git puedes:

- Guardar cambios progresivos en tu trabajo
- Colaborar con otras personas sin perder el control de versiones
- Descargar (clonar) proyectos públicos desde plataformas como GitHub

En este notebook usamos `git clone` para copiar un repositorio que contiene bases de datos educativas en formato SQLite.

### ¿Qué es SQL?

SQL (Structured Query Language) es un lenguaje utilizado para consultar y manipular bases de datos. Permite extraer, filtrar, combinar y ordenar información de forma estructurada.

En este notebook usamos SQL para obtener datos de una base de datos SQLite. Algunas cláusulas clave que utilizamos son:

- `SELECT`: indica qué columnas queremos ver
- `FROM`: especifica de qué tabla se obtienen los datos
- `JOIN`: combina información de varias tablas relacionadas
- `ON`: define la condición de unión entre tablas
- `LIMIT`: restringe la cantidad de filas que se muestran (opcional)

Ejemplo usado:
```sql
SELECT carat, price, cut
FROM Observation
JOIN Cut ON Observation.cut_id = Cut.cut_id

In [1]:
# Conectarse a la base de datos e importar librerías

import sqlite3
import requests
import pandas as pd

url = "https://raw.githubusercontent.com/davidjamesknight/SQLite_databases_for_learning_data_science/main/diamonds.db"
r = requests.get(url)

with open("diamonds.db", "wb") as f:
    f.write(r.content)

db = sqlite3.connect("diamonds.db")
cursor = db.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tablas = cursor.fetchall()

print("Tablas encontradas:")
for tabla in tablas:
    print(tabla[0])

Tablas encontradas:
Observation
Cut
Color
Clarity


In [2]:
# TODORecorrer cada tabla y mostrar sus columnas
for tabla in tablas:
    nombre_tabla = tabla[0]  # Extraer el nombre como cadena
    print(f"\nColumnas en la tabla '{nombre_tabla}':")

    # Usar PRAGMA para obtener información de las columnas
    cursor.execute(f"PRAGMA table_info('{nombre_tabla}');")
    columnas = cursor.fetchall()

    # Mostrar nombre y tipo de cada columna
    for columna in columnas:
        print(f" - {columna[1]} ({columna[2]})")  # columna[1] = nombre, columna[2] = tipo


Columnas en la tabla 'Observation':
 - carat (FLOAT)
 - depth (FLOAT)
 - table (FLOAT)
 - price (BIGINT)
 - x (FLOAT)
 - y (FLOAT)
 - z (FLOAT)
 - cut_id (BIGINT)
 - color_id (BIGINT)
 - clarity_id (BIGINT)

Columnas en la tabla 'Cut':
 - cut_id (BIGINT)
 - cut (TEXT)

Columnas en la tabla 'Color':
 - color_id (BIGINT)
 - color (TEXT)

Columnas en la tabla 'Clarity':
 - clarity_id (BIGINT)
 - clarity (TEXT)


# Generar primer SQL query.

In [3]:
# TODO: Crear la consulta SQL

sql = """
SELECT 
    O.carat,
    O.price, 
    O.depth,
    O."table",
    O.x,
    O.y,
    O.z,
    C.cut,
    Co.color,
    Cl.clarity
FROM
    Observation AS O 
    JOIN
        Cut AS C 
        ON O.cut_id = C.cut_id
    JOIN
        Color AS Co
        ON O.color_id = Co.color_id
    JOIN
        Clarity AS Cl
        ON O.clarity_id = Cl.clarity_id
"""

# TODO: Almacenar datos en dataframe (df)
df = pd.read_sql_query(sql, db)
display(df)

,carat,price,depth,table,x,y,z,cut,color,clarity
0,0.23,326,61.5,55.0,3.95,3.98,2.43,Ideal,E,SI2
1,0.21,326,59.8,61.0,3.89,3.84,2.31,Premium,E,SI1
2,0.23,327,56.9,65.0,4.05,4.07,2.31,Good,E,VS1
3,0.29,334,62.4,58.0,4.20,4.23,2.63,Premium,I,VS2
4,0.31,335,63.3,58.0,4.34,4.35,2.75,Good,J,SI2
...,...,...,...,...,...,...,...,...,...,...
53935,0.72,2757,60.8,57.0,5.75,5.76,3.50,Ideal,D,SI1
53936,0.72,2757,63.1,55.0,5.69,5.75,3.61,Good,D,SI1
53937,0.70,2757,62.8,60.0,5.66,5.68,3.56,Very Good,D,SI1
53938,0.86,2757,61.0,58.0,6.15,6.12,3.74,Premium,H,SI2


### Identificar tipos de datos

In [4]:
# TODO: Identifica tipos de datos
df.dtypes

carat      float64
price        int64
depth      float64
table      float64
x          float64
y          float64
z          float64
cut         object
color       object
clarity     object
dtype: object

In [5]:
# TODO: Separa por numéricas y categóricas
numericas = df.select_dtypes(
    include=['float64','int64']
).columns.tolist()

numericas

categoricas = df.select_dtypes(
    include=['object']
).columns.tolist()

print("Variables numericas:", numericas)
print("Variables categóricas:", categoricas)

Variables numericas: ['carat', 'price', 'depth', 'table', 'x', 'y', 'z']
Variables categóricas: ['cut', 'color', 'clarity']


### Escala de medición de las variables

| Variable         | Tipo de dato | Escala de medición | Justificación |
|------------------|--------------|---------------------|----------------|
| `carat`          | Numérica     | Razón               | Tiene cero absoluto, se puede multiplicar/dividir |
| `price`          | Numérica     | Razón               | Representa valor monetario, cero tiene significado |
| `x`, `y`, `z`     | Numérica     | Razón               | Medidas físicas, cero indica ausencia |
| `depth`          | Numérica     | Intervalo           | Porcentaje relativo, no tiene cero absoluto claro |
| `"table"`        | Numérica     | Intervalo           | Porcentaje relativo, no tiene cero absoluto claro |
| `cut`            | Categórica   | Ordinal             | Tiene orden lógico: Fair < Good < Very Good < Ideal < Premium |
| `color`          | Categórica   | Ordinal o Nominal           | Escala gemológica: D (mejor) a J (peor) |
| `clarity`        | Categórica   | Ordinal o Nominal           | Escala gemológica: FL > IF > VVS1 > ... > I3 |


### Estadísticas Descriptivas

In [6]:
# TODO: Obten estadísticas desceiptivas por medio del metodo de pandas
df.describe()

,carat,price,depth,table,x,y,z
count,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000
mean,0.797940,3932.799722,61.749405,57.457184,5.731157,5.734526,3.538734
std,0.474011,3989.439738,1.432621,2.234491,1.121761,1.142135,0.705699
min,0.200000,326.000000,43.000000,43.000000,0.000000,0.000000,0.000000
25%,0.400000,950.000000,61.000000,56.000000,4.710000,4.720000,2.910000
50%,0.700000,2401.000000,61.800000,57.000000,5.700000,5.710000,3.530000
75%,1.040000,5324.250000,62.500000,59.000000,6.540000,6.540000,4.040000
max,5.010000,18823.000000,79.000000,95.000000,10.740000,58.900000,31.800000


In [7]:
# TODO: Obten conteo de valores para cada categoría de las variables catgeóricas

- ¿Qué tipo de codificación sería más adecuada para cada variable categórica?


### Correlaciones entre variables numéricoas

In [8]:
# TODO: Crear Matriz de Correlación con plotly (sólo la primera vez)
import plotly.figure_factory as ff

- ¿Qué variables parecen tener una relación fuerte con el precio?

### Visualizaciones con plotly

In [9]:
# TODO: Crear Scatter Plot
import plotly.express as px

### Boxplot pot tipo de corte

In [10]:
# TODO: Crear boxplot por tipo de corte

- ¿Qué tipo de corte tiene mayor dispersión en precios?

### Histogramas de variables numericas

In [11]:
# TODO: Histogramas de variables numéricas

- ¿Qué variables numéricas muestran distribución sesgada?


### Mapa de calor con correlaciones

### Detección de outliers utilizando RIC (IQR)

In [12]:
# TODO: Obtener numero de outliers por variable numérica

### Análisis de Distribución

In [13]:
# TODO: Obtener sesgo y curtosis de variables numéricas

### Transformación de variables categóricas

In [14]:
# TODO: Label Encoding con sklearn

El órden lógico de esta columna es Fair < Good < Very Good < Ideal < Premium

In [15]:
# TODO: Label Encoding Manual

In [16]:
# TODO: One-Hot Encoding

In [17]:
# TODO: Target Encoder
import category_encoders as ce

### Escalamiento de Datos

### ¿Qué es el escalamiento de datos?

El escalamiento de datos es una técnica que ajusta los valores numéricos para que estén en rangos comparables. Esto es útil cuando usamos algoritmos que son sensibles a las magnitudes de los datos (como regresiones o clustering).

A continuación, se explican tres métodos comunes:

---

#### 🔹 MinMaxScaler

- Ajusta los valores para que estén entre un mínimo y un máximo (por defecto, entre 0 y 1).
- Fórmula:  
  $$
  X_{\text{escalado}} = \frac{X - X_{\text{min}}}{X_{\text{max}} - X_{\text{min}}}
  $$
- Útil cuando queremos conservar la forma original de la distribución.

---

#### 🔹 StandardScaler

- Centra los datos en 0 y los escala para que tengan una desviación estándar de 1.
- Fórmula:  
  $$
  X_{\text{escalado}} = \frac{X - \mu}{\sigma}
  $$
  donde $\mu$ es la media y $\sigma$ la desviación estándar.
- Ideal cuando los datos tienen una distribución aproximadamente normal.

---

En resumen: estos métodos ayudan a que los datos sean más comparables y útiles para análisis estadísticos o modelos de machine learning.

### ¿Se pueden combinar distintos escaladores?

Sí, es válido aplicar diferentes métodos de escalamiento a distintas columnas, especialmente cuando:

- Las variables tienen **naturaleza distinta** (por ejemplo, ingresos vs. proporciones).
- Algunas columnas tienen **valores extremos (outliers)** y otras no.
- Quieres conservar la **forma original** de ciertas distribuciones (MinMaxScaler) pero estandarizar otras (StandardScaler).

Ejemplo práctico:
- Usar `StandardScaler` en columnas como "ingresos" o "edad", que tienen distribución normal.
- Usar `MinMaxScaler` en columnas como "porcentaje de cumplimiento" o "calificaciones", que ya están en rangos definidos.

---

### Consideraciones importantes

- Si usas modelos **basados en distancia** (como KNN o clustering), asegúrate de que las escalas no generen sesgos. En ese caso, es mejor que todas las variables estén en rangos comparables.
- Para modelos como **árboles de decisión o random forest**, el escalamiento no es necesario, ya que no dependen de magnitudes.

---

En resumen: puedes escalar columnas de forma diferente, pero asegúrate de que tenga sentido para el tipo de análisis o modelo que estás usando.

# Pruebas de Hipótesis: Una y Dos Medias

Las **pruebas de hipótesis** nos ayudan a tomar decisiones sobre una población usando datos de una muestra. En particular, podemos comparar medias para saber si hay diferencias significativas o si los resultados podrían deberse al azar.

## ¿Qué es el estadístico t y el valor p?

- **Estadístico t:** Es una medida que compara la diferencia observada entre medias (o entre una media y un valor de referencia) con la variabilidad de los datos. Nos dice cuántas "desviaciones estándar" está la diferencia observada respecto a lo que esperaríamos por azar.
- **Valor p:** Es la probabilidad de obtener un resultado igual o más extremo que el observado, suponiendo que la hipótesis nula es cierta. Si el valor p es pequeño (por ejemplo, menor a 0.05), consideramos que la diferencia es significativa.

### Fórmulas

**Para una muestra:**

$$
t = \frac{\bar{x} - \mu_0}{s / \sqrt{n}}
$$

donde:

- $\bar{x}$ = media muestral  
- $\mu_0$ = media bajo la hipótesis nula  
- $s$ = desviación estándar muestral  
- $n$ = tamaño de la muestra  

**Para dos muestras independientes:**

$$
t = \frac{\bar{x}_1 - \bar{x}_2}{\sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}}
$$

donde los subíndices 1 y 2 corresponden a cada grupo.

---

In [18]:
# TODO: Prueba de hipótesis para una media (t-test de una muestra)


## 2. Prueba de Hipótesis para dos Medias

Se utiliza para comparar si las medias de dos grupos son iguales.

**Ejemplo:**  
¿El precio promedio de los diamantes con corte 'Ideal' es igual al de los de corte 'Premium'?

- **Hipótesis nula ($H_0$):** $\mu_1 = \mu_2$
- **Hipótesis alternativa ($H_1$):** $\mu_1 \neq \mu_2$

In [19]:
# TODO: Prueba de hipótesis para dos medias independientes (t-test de dos muestras)